# Create Training Labels from the Critical Mineral Deposits Database

In [ ]:
import os
import numpy as np
import geopandas as gpd
from pathlib import Path

import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

from beak.utilities.io import load_dataset
from beak.utilities.conversions import create_binary_raster

import numpy as np
import pandas as pd
from shapely.geometry import shape
from shapely.wkt import loads
import rasterio


# Definitions

In [ ]:
# Set base paths and files
BASE_PATH = files("beak.data")

EPSG_CODE = 102008
RESOLUTION = 500
BASE_SPATIAL = str(EPSG_CODE) + "_" + str(RESOLUTION)
BASE_EXTENT = "mama_nico_upmidwest"

BASE_RASTER = BASE_PATH / "BASE_RASTERS" / str("EPSG_" + str(EPSG_CODE) + "_RES_" + str(RESOLUTION) + "_" + BASE_EXTENT + ".tif")
base_raster = rasterio.open(BASE_RASTER)

# Points file and query to select relevant mineral occurences
PATH_LABELS = BASE_PATH / "RAW" / "mineral_deposits" / "Magmatic_Cobalt_Nickel" / "TA2_Pre_Hack_12M" / "set_20240609" / "nickel_mineral_site_data.csv"
PATH_LABELS_TA2_MMS_HM9 = BASE_PATH / "RAW" / "mineral_deposits" / "Magmatic_Cobalt_Nickel" / "TA2_and_Rosera_for_hack_9m" / "cobalt_mafic_magmatic_systems_merged_with_ta2_hm6_labels.shp"
SQL_QUERY = "country == 'United States'"

# Set the output file
PATH_ROOT = BASE_PATH / "PROCESSED" / str("regional" + "_" + BASE_EXTENT + "_" + BASE_SPATIAL)
PATH_EXPORT = PATH_ROOT / "labels" / str("TA2_240609_FILTERED_HM9_TA2_MMS" + ".tif")
OUT_FILE = PATH_EXPORT
OUT_GPKT = str(PATH_LABELS.parent / str(PATH_LABELS.stem)) + ".gpkg"

print(f"Output file: {OUT_FILE}")


# **Helper**

In [ ]:
import matplotlib.pyplot as plt
def plot_histogram(df, column, num_bins=100):
    df['top1_deposit_classification_confidence'].hist(bins=num_bins)
    plt.xlabel(column)
    plt.ylabel('Frequency')
    plt.title(f"Histogram of {column}")
    plt.show()

In [ ]:
def count_values(df, column):    
    values = df[column].unique()
    values_df = pd.DataFrame(values, columns=[column])
    counts = df[column].value_counts()
    values_with_counts = counts.reset_index()
    values_with_counts.columns = [column, 'count']
    
    return values_with_counts


# **Load and initially clean data**

In [ ]:
mineral_sites = gpd.read_file(PATH_LABELS)

mineral_sites = load_dataset(PATH_LABELS).query(SQL_QUERY)
mineral_sites = mineral_sites.dropna(subset=["loc_wkt"])
mineral_sites["geometry"] = mineral_sites["loc_wkt"].apply(loads)
mineral_sites = gpd.GeoDataFrame(mineral_sites, geometry="geometry")
mineral_sites = mineral_sites.explode(ignore_index=True)
mineral_sites = mineral_sites.drop_duplicates(subset=["geometry"])
mineral_sites = mineral_sites.set_crs("EPSG:4326")

mineral_sites

In [ ]:
plot_histogram(mineral_sites, "top1_deposit_classification_confidence")

# Export to GeoPackages

## Filtered Points

In [ ]:
query = "top1_deposit_type in ('U-M layered intrusion nickel- copper-PGE','U-M intrusion nickel-copper- PGE','U-M conduit nickel-copper- PGE') and top1_deposit_classification_confidence >= 0.5 and ms_type in ('Past Producer', 'Prospect', 'Producer') and ms_rank in ('A', 'B', 'C')"

mineral_sites_filtered = mineral_sites.query(query)
mineral_sites_filtered

In [ ]:
mineral_sites_filtered.to_file(OUT_GPKT, layer="mineral_sites_filtered", driver="GPKG")

## Add TA2 HM9 Data
Data originally from HM6 and re-used at HM9

In [21]:
# Labels MC
labels_ta2_mms_hm9 = gpd.read_file(PATH_LABELS_TA2_MMS_HM9)
labels_ta2_mms_hm9 = labels_ta2_mms_hm9.to_crs(mineral_sites.crs)
labels_ta2_mms_hm9

,Deposit,Lat_WGS84,Long_WGS84,Source_s,Links,grade,Source,fid,Unnamed_ 0,ms.value,...,oms.value,Unnamed__1,source_id,prediction,ms_record_,ms_source_,site_hyper,layer,path,geometry
0,Maturi,47.80480,-91.72869,Barber and others (2014); Burger and others (2...,https://doi.org/10.5066/P9V74HIU,None,Critical mineral deposits release,NaN,NaN,None,...,None,NaN,None,NaN,None,None,None,cobalt_mafic_magmatic_systems,Z:/2023/0051-0100/20230082_DARPA_CriticalMAAS_...,MULTIPOINT ((-91.72869 47.80480))
1,Mirror Harbor,57.78550,-136.30820,ARDF (1996),https://mrdata.usgs.gov/ardf/show-ardf.php?ard...,None,Critical mineral deposits release,NaN,NaN,None,...,None,NaN,None,NaN,None,None,None,cobalt_mafic_magmatic_systems,Z:/2023/0051-0100/20230082_DARPA_CriticalMAAS_...,MULTIPOINT ((-136.30820 57.78550))
2,Mesaba project (Minnamax; Babbitt),47.63166,-91.88323,MRDS (2005); Listerud and Meineke (1977); Welh...,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...,None,Critical mineral deposits release,NaN,NaN,None,...,None,NaN,None,NaN,None,None,None,cobalt_mafic_magmatic_systems,Z:/2023/0051-0100/20230082_DARPA_CriticalMAAS_...,MULTIPOINT ((-91.88323 47.63166))
3,"Iron Mountain, HGR nickel-copper-PGE deposit",45.40480,-110.04601,Childs and Armitage (2021),None,None,Critical mineral deposits release,NaN,NaN,None,...,None,NaN,None,NaN,None,None,None,cobalt_mafic_magmatic_systems,Z:/2023/0051-0100/20230082_DARPA_CriticalMAAS_...,MULTIPOINT ((-110.04601 45.40480))
4,"Iron Mountain, CZ nickel-copper-PGE deposit",45.40798,-110.07355,Childs and Armitage (2021),None,None,Critical mineral deposits release,NaN,NaN,None,...,None,NaN,None,NaN,None,None,None,cobalt_mafic_magmatic_systems,Z:/2023/0051-0100/20230082_DARPA_CriticalMAAS_...,MULTIPOINT ((-110.07355 45.40798))
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3693,None,NaN,NaN,None,None,None,MRDS,27357.0,27356.0,https://minmod.isi.edu/resource/mrds10112530,...,None,1464.0,10112530,841.0,10112530,MRDS,https://minmod.isi.edu/resource/841,hypersite_nickel_us_data_filtered,S:/Projekte/20230082_DARPA_CriticalMAAS_TA3/Be...,MULTIPOINT ((-155.59683 62.89525))
3694,None,NaN,NaN,None,None,None,MRDS,27358.0,27357.0,https://minmod.isi.edu/resource/mrds10112530,...,None,1464.0,10112530,841.0,10112530,MRDS,https://minmod.isi.edu/resource/841,hypersite_nickel_us_data_filtered,S:/Projekte/20230082_DARPA_CriticalMAAS_TA3/Be...,MULTIPOINT ((-155.59683 62.89525))
3695,None,NaN,NaN,None,None,None,MRDS,27359.0,27358.0,https://minmod.isi.edu/resource/mrds10112530,...,None,1464.0,10112530,841.0,10112530,MRDS,https://minmod.isi.edu/resource/841,hypersite_nickel_us_data_filtered,S:/Projekte/20230082_DARPA_CriticalMAAS_TA3/Be...,MULTIPOINT ((-155.59683 62.89525))
3696,None,NaN,NaN,None,None,None,MRDS,27360.0,27359.0,https://minmod.isi.edu/resource/mrds10112530,...,None,1464.0,10112530,841.0,10112530,MRDS,https://minmod.isi.edu/resource/841,hypersite_nickel_us_data_filtered,S:/Projekte/20230082_DARPA_CriticalMAAS_TA3/Be...,MULTIPOINT ((-155.59683 62.89525))


In [31]:
labels_ta2_mms_hm9 = labels_ta2_mms_hm9.geometry.copy()
labels_ta2_mms_hm9.to_file(OUT_GPKT, layer="mafic_magmatic_systems_and_ta2_from_hm9", driver="GPKG", overwrite=True)


In [35]:
# Merge
labels = pd.concat([mineral_sites_filtered.geometry, 
                    labels_ta2_mms_hm9.geometry], ignore_index=True)


# Create Labels

In [37]:
data = labels.copy()
data = data.explode(ignore_index=True)
data = data[~data.is_empty]
data = data.reset_index(drop=True)
data = data.to_crs(base_raster.crs)


In [38]:
labels_array = create_binary_raster(data, base_raster, all_touched=False, same_shape=True, fill_negatives=True, out_file=None)
print(f"Number of positive training labels: {np.sum(labels_array==1)}")

Number of positive training labels: 43
